# async io practice

In [2]:
import asyncio
import time


In [3]:
# --------helper functions--------
def blocking_work(seconds, label):
    #simulate blocking sync code
    time.sleep(seconds)
    return f"{label} done in {seconds} seconds"

async def worker(name, delay):
    await asyncio.sleep(delay)
    return f"worker {name} done in {delay} seconds"

async def slow_operation(delay):
    await asyncio.sleep(delay)
    return f"slow operation done in {delay} seconds"

In [5]:
# --------main demo coode--------
async def asyncio_demo():
    print("Starting asyncio demo...")

    loop = asyncio.get_running_loop()
    me = asyncio.current_task()
    tasks_now = asyncio.all_tasks()
    another_task = asyncio.all_tasks()

    print("Running loop:",loop)
    print("Current task:", me.get_name() if hasattr(me, 'get_name') else me)
    print("Total tasks currently alive:", len(tasks_now))

    print("\n======== create+task + task methiods demo ========")
    t1 = asyncio.create_task(worker("A", 2), name="WorkerA")
    t2 = asyncio.create_task(worker("B", 3), name="WorkerB")
    print("t1 done?", t1.done())
    print("t2 done?", t2.done())


    await asyncio.sleep(1.5) # wait a bit before checking again
    print("After 1.5 seconds:")
    print("t1 done?", t1.done())
    print("t2 done?", t2.done())


    if t1.done():
        print("t1 result:", t1.result())

    print("\n======== gather demo ========")
    results = await asyncio.gather(worker("C", 1), worker("D", 2))
    print("Gather results:", results)

    print("\n======== wait demo ========")
    t3 = asyncio.create_task(worker("E", 2), name="wait-1")
    t4 = asyncio.create_task(worker("F", 4), name="wait-2")

    done, pending = await asyncio.wait({t3,t4},timeout=1)
    print("done count:", len(done),"pending count:",len(pending))

    for d in done:
        print("done task result:",d.result())
    for p in pending:
        print("cancelling pending task:",p.get_name() if hasattr(p,"get_name") else p)
        p.cancel()
        try:
            await p
        except asyncio.CancelledError:
            print("pending task canceled")
    
    print("\n======wait for time out=====")
    try:
        await asyncio.wait_for(slow_operation(3),timeout=0.8)
    except asyncio.TimeoutError:
        print("wait_for timed out as expected")

    print("\n========sheild=======")
    protected = asyncio.create_task(slow_operation(1.5),name="protected_task")
    try:
        await asyncio.wait_for(asyncio.shield(protected),timeout=0.5)
    except asyncio.TimeoutError:
        print("outer timeout happpend,but shield protected inner task")
    print("protected done right after timeout?",protected.done())
    print("protected final result:",await protected)

    print("\n============== to_thread===========")

    thread_result = await asyncio.to_thread(blocking_work,1,"job-1")
    print(thread_result)

    print("\n==========task.cancel + exception======")
    t_cancel = asyncio.create_task(slow_operation(3),name="cancel-me")
    await asyncio.sleep(0.4)
    t_cancel.cancel()
    print("cancel requested; done?",t_cancel.done())
    try:
        await t_cancel
    except asyncio.CancelledError:
        print("task cancelled correctly")
    print("done now?", t_cancel.done())
    print("exception on cancelled task:")
    try:
        print(t_cancel.exception())
    except asyncio.CancelledError as e:
        print("CancelledError raised when reading exception():", e)

    print("\n=== Queue producer/consumer ===")

    queue = asyncio.Queue()

    async def producer():
        for i in range(5):
            item = f"item-{i}"
            await queue.put(item) # Put item in the queue
            print(f"Produced {item}")
            await asyncio.sleep(0.2)
        await queue.put(None)  # Sentinel to signal consumer to stop

    async def consumer():
        while True:
            item = await queue.get() #queue.get() will wait until an item is available
            if item is None:
                queue.task_done() # Mark the sentinel as processed
                break
            print(f"Consumed {item}")
            await asyncio.sleep(0.3)
            queue.task_done() # Mark the item as processed

    prod_task = asyncio.create_task(producer(), name="producer")
    cons_task = asyncio.create_task(consumer(), name="consumer")
    await asyncio.gather(prod_task, cons_task)
    await queue.join() # Wait until all items have been processed
    print("queue processing complete.")

await asyncio_demo()

Starting asyncio demo...
Running loop: <_WindowsSelectorEventLoop running=True closed=False debug=False>
Current task: Task-44
Total tasks currently alive: 3

======== create+task + task methiods demo ========
t1 done? False
t2 done? False
After 1.5 seconds:
t1 done? False
t2 done? False

======== gather demo ========
Gather results: ['worker C done in 1 seconds', 'worker D done in 2 seconds']

======== wait demo ========
done count: 0 pending count: 2
cancelling pending task: wait-2
pending task canceled
cancelling pending task: wait-1
pending task canceled

======wait for time out=====
wait_for timed out as expected

========sheild=======
outer timeout happpend,but shield protected inner task
protected done right after timeout? False
protected final result: slow operation done in 1.5 seconds

============== to_thread===========
job-1 done in 1 seconds

==========task.cancel + exception======
cancel requested; done? False
task cancelled correctly
done now? True
exception on cancelled 